# Basic connection creation

This notebook demonstrates the creation of APIM connection objects using static models. 

You must create and populate an .env-setup file. A sample is provided in the env-setup-sample file.

## Setup logging

In [ ]:
import logging
import sys

def configure_logging(level="DEBUG"):
    """This function configures logging for code being run based on the specified level.
    Args:
        level (str): The logging level to use (e.g., "DEBUG", "INFO", "WARNING", "ERROR", "CRITICAL").
    """

    try:
        # Convert the level string to uppercase so it matches what the logging library expects
        logging_level = getattr(logging, level.upper(), None)

        # Setup a logging format
        logging.basicConfig(
            level=logging_level,
            format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
            handlers=[logging.StreamHandler(sys.stdout)]
        )
    except Exception as e:
        print(f"Failed to set up logging: {e}", file=sys.stderr)
        sys.exit(1) 

configure_logging("ERROR")

## Get user access token for ARM REST API

In [ ]:
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

load_dotenv(".env-setup", override=True)

# Get a token for Microsoft Graph with the logged in user
credential = DefaultAzureCredential()
scopes = ["https://management.azure.com/.default"]

user_token = credential.get_token(*scopes)

## Create re-usable functions to create static model connections

This assumes you have deployed 4o. Customize this as needed for other models

In [ ]:
import requests
import os

def create_apim_connection_account(subscription: str, resource_group_name: str, foundry_account_name: str, connection_name: str, apim_url: str, token: str):
    url = f"https://management.azure.com/subscriptions/{subscription}/resourceGroups/{resource_group_name}/providers/Microsoft.CognitiveServices/accounts/{foundry_account_name}/connections/{connection_name}"
    api_version = "2025-10-01-preview"

    body = {
        "properties": {
            "category": "ApiManagement",
            "target": apim_url,
            "authType": "AAD",
            "isSharedToAll": True,
             "metadata": {
                    "deploymentInPath": "true",
                    "inferenceAPIVersion": "2025-03-01-preview",
                    "models": "[{\"name\":\"gpt-4o\",\"properties\":{\"model\":{\"name\":\"gpt-4o\",\"version\":\"2024-08-06\",\"format\":\"OpenAI\"}}}]"
            }
        }
    }


    response = requests.put(
        url = url,
        params = {"api-version": api_version},
        headers = 
        {
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json"
        },
        json = body
    )

    print(response.status_code)
    print(response.text)

def delete_apim_connection_account(subscription: str, resource_group_name: str, foundry_account_name: str, connection_name: str, token: str):
    url = f"https://management.azure.com/subscriptions/{subscription}/resourceGroups/{resource_group_name}/providers/Microsoft.CognitiveServices/accounts/{foundry_account_name}/connections/{connection_name}"
    api_version = "2025-10-01-preview"

    response = requests.delete(
        url = url,
        params = {"api-version": api_version},
        headers = 
        {
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json"
        }
    )

    print(response.status_code)
    print(response.text)

def create_apim_connection_project(subscription: str, resource_group_name: str, foundry_account_name: str, project_name: str, connection_name: str, apim_url: str, token: str):
    url = f"https://management.azure.com/subscriptions/{subscription}/resourceGroups/{resource_group_name}/providers/Microsoft.CognitiveServices/accounts/{foundry_account_name}/projects/{project_name}/connections/{connection_name}"
    api_version = "2025-10-01-preview"

    body = {
        "properties": {
            "category": "ApiManagement",
            "target": apim_url,
            "authType": "AAD",
            "isSharedToAll": True,
             "metadata": {
                    "deploymentInPath": "true",
                    "inferenceAPIVersion": "2025-03-01-preview",
                    "models": "[{\"name\":\"gpt-4o\",\"properties\":{\"model\":{\"name\":\"gpt-4o\",\"version\":\"2024-08-06\",\"format\":\"OpenAI\"}}}]"
            }
        }
    }


    response = requests.put(
        url = url,
        params = {"api-version": api_version},
        headers = 
        {
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json"
        },
        json = body
    )

    print(response.status_code)
    print(response.text)

def delete_apim_connection_project(subscription: str, resource_group_name: str, foundry_account_name: str, project_name: str, connection_name: str, token: str):
    url = f"https://management.azure.com/subscriptions/{subscription}/resourceGroups/{resource_group_name}/providers/Microsoft.CognitiveServices/accounts/{foundry_account_name}/projects/{project_name}/connections/{connection_name}"
    api_version = "2025-10-01-preview"

    response = requests.delete(
        url = url,
        params = {"api-version": api_version},
        headers = 
        {
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json"
        }
    )

    print(response.status_code)
    print(response.text)

def list_account_connections(subscription: str, resource_group_name: str, foundry_account_name: str, token: str):
    url = f"https://management.azure.com/subscriptions/{subscription}/resourceGroups/{resource_group_name}/providers/Microsoft.CognitiveServices/accounts/{foundry_account_name}/connections"
    api_version = "2025-10-01-preview"

    response = requests.get(
        url = url,
        params = {"api-version": api_version},
        headers = 
        {
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json"
        }
    )

    print(f"Status: {response.status_code}")
    data = response.json()
    for conn in data.get("value", []):
        props = conn.get("properties", {})
        metadata = props.get("metadata", {})
        print(f"\n  Name     : {conn.get('name')}")
        print(f"  Category : {props.get('category')}")
        print(f"  AuthType : {props.get('authType')}")
        print(f"  Target   : {props.get('target')}")
        print(f"  Shared   : {props.get('isSharedToAll')}")
        if metadata:
            print(f"  Metadata :")
            for k, v in metadata.items():
                print(f"    {k} : {v}")

def list_project_connections(subscription: str, resource_group_name: str, foundry_account_name: str, project_name: str, token: str):
    url = f"https://management.azure.com/subscriptions/{subscription}/resourceGroups/{resource_group_name}/providers/Microsoft.CognitiveServices/accounts/{foundry_account_name}/projects/{project_name}/connections"
    api_version = "2025-10-01-preview"

    response = requests.get(
        url = url,
        params = {"api-version": api_version},
        headers = 
        {
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json"
        }
    )

    print(f"Status: {response.status_code}")
    data = response.json()
    for conn in data.get("value", []):
        props = conn.get("properties", {})
        metadata = props.get("metadata", {})
        print(f"\n  Name     : {conn.get('name')}")
        print(f"  Category : {props.get('category')}")
        print(f"  AuthType : {props.get('authType')}")
        print(f"  Target   : {props.get('target')}")
        print(f"  Shared   : {props.get('isSharedToAll')}")
        if metadata:
            print(f"  Metadata :")
            for k, v in metadata.items():
                print(f"    {k} : {v}")


## List existing connections

In [ ]:
print("\nAccount Connections:")

list_account_connections(
    subscription=os.getenv("SUBSCRIPTION_ID"),
    resource_group_name=os.getenv("FOUNDRY_RESOURCE_GROUP"),
    foundry_account_name=os.getenv("FOUNDRY_ACCOUNT_NAME"),
    token=user_token.token
)

print("\nProject Connections:")

list_project_connections(
    subscription=os.getenv("SUBSCRIPTION_ID"),
    resource_group_name=os.getenv("FOUNDRY_RESOURCE_GROUP"),
    foundry_account_name=os.getenv("FOUNDRY_ACCOUNT_NAME"),
    project_name=os.getenv("FOUNDRY_PROJECT_NAME"),
    token=user_token.token
)

## Create account connection

In [ ]:
account_connection = create_apim_connection_account(
    subscription=os.getenv("SUBSCRIPTION_ID"),
    resource_group_name=os.getenv("FOUNDRY_RESOURCE_GROUP"),
    foundry_account_name=os.getenv("FOUNDRY_ACCOUNT_NAME"),
    connection_name="apim-connection-account-static",
    apim_url=f"https://{os.getenv("APIM_NAME")}/openai",
    token=user_token.token
)   

## Create project connection

In [ ]:
create_apim_connection_project(
    subscription=os.getenv("SUBSCRIPTION_ID"),
    resource_group_name=os.getenv("FOUNDRY_RESOURCE_GROUP"),
    foundry_account_name=os.getenv("FOUNDRY_ACCOUNT_NAME"),
    project_name=os.getenv("PROJECT_NAME"),
    connection_name="apim-connection-project-static",
    apim_url=os.getenv("APIM_URL"),
    token=user_token.token
)


## Cleanup connections

In [ ]:
delete_apim_connection_account(
    subscription=os.getenv("SUBSCRIPTION_ID"),
    resource_group_name=os.getenv("FOUNDRY_RESOURCE_GROUP"),
    foundry_account_name=os.getenv("FOUNDRY_ACCOUNT_NAME"),
    connection_name="apim-connection-account-static",
    token=user_token.token
)

delete_apim_connection_project(
    subscription=os.getenv("SUBSCRIPTION_ID"),
    resource_group_name=os.getenv("FOUNDRY_RESOURCE_GROUP"),
    foundry_account_name=os.getenv("FOUNDRY_ACCOUNT_NAME"),
    project_name=os.getenv("FOUNDRY_PROJECT_NAME"),
    connection_name="apim-connection-project-static",
    token=user_token.token
)